# End-to-End Machine Learning Project: Breast Cancer Diagnosis

This notebook walks through a complete supervised-learning project from raw data to a saved, deployable model. It mirrors how a real project is run: frame the problem, explore the data, engineer features, compare models, tune the winner, evaluate honestly on held-out data, and write up the findings.

## Problem framing

**What we predict.** Given 30 numeric measurements computed from a digitized image of a fine-needle aspirate of a breast mass (cell radius, texture, concavity, and so on), classify the tumor as **malignant** or **benign**.

**Why it matters.** This is a decision-support setting. A model that reliably flags malignant tumors can help prioritize which patients need urgent follow-up, while confidently-benign cases reduce unnecessary anxiety and cost.

**The metric we optimize, and why.** We select models by **ROC AUC** during cross-validation. AUC is threshold-independent and robust to the mild class imbalance (about 37% malignant), so it measures how well the model *ranks* malignant above benign regardless of where we later set the decision cut-off. At final evaluation we also report **recall on the malignant class (sensitivity)**, because the most expensive mistake here is a missed cancer (a false negative), not a false alarm.

Everything runs offline on the Wisconsin Breast Cancer dataset bundled inside scikit-learn.

In [ ]:
import numpy as np                              # numerical arrays and math
import pandas as pd                             # tabular data handling (DataFrame)
import matplotlib.pyplot as plt                 # base plotting
import seaborn as sns                           # statistical visualizations on top of matplotlib
import joblib                                   # persist (save/load) the trained pipeline to disk

from sklearn.datasets import load_breast_cancer            # local, offline classification dataset
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline                      # chain preprocessing + model into one object
from sklearn.preprocessing import StandardScaler, FunctionTransformer  # scaling + custom feature step
from sklearn.linear_model import LogisticRegression        # linear baseline classifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier  # tree ensembles
from sklearn.svm import SVC                                 # kernel support vector classifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, roc_auc_score, ConfusionMatrixDisplay)

# Reproducibility: fix the global seed so splits/models give the same numbers each run.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style='whitegrid')               # consistent, clean look for all seaborn plots
pd.set_option('display.width', 120)            # wider console output for tables

## 1. Load the data

The dataset ships with scikit-learn, so there is nothing to download. We load it into a pandas DataFrame with readable column names and attach the target. Note the deliberate label flip so that `1 = malignant` is the positive class we care about detecting.

In [ ]:
# Load the Wisconsin Breast Cancer dataset that ships INSIDE scikit-learn (no download needed).
raw = load_breast_cancer()

# raw.data -> (569, 30) feature matrix; raw.feature_names -> the 30 column names.
feature_names = list(raw.feature_names)

# Build a tidy pandas DataFrame: one column per measurement, human-readable names.
df = pd.DataFrame(raw.data, columns=feature_names)

# IMPORTANT label convention. In sklearn this dataset codes 0 = malignant, 1 = benign.
# For a diagnosis tool the event we care about CATCHING is the malignant tumor, so we
# re-encode the target as 1 = malignant (positive class), 0 = benign. This makes
# 'recall' read directly as 'fraction of real cancers we flagged'.
df['target'] = (raw.target == 0).astype(int)

# Quick peek at the first rows to confirm the frame looks sane.
df.head()

## 2. Exploratory data analysis (EDA)

Before modelling we check the shape and types, confirm there are no missing values to impute, measure the class balance, and summarize a few features. These checks decide what preprocessing we actually need.

In [ ]:
# ---- Basic structural checks: the first things to look at on any new dataset ----
print('shape (rows, cols):', df.shape)                     # how much data do we have?
print()
print('dtype counts:')
print(df.dtypes.value_counts())                            # all numeric here -> no categorical encoding needed
print()
print('missing values across the whole frame:', int(df.isna().sum().sum()))  # 0 -> nothing to impute

# ---- Target balance: is one class much rarer than the other? ----
counts = df['target'].value_counts().sort_index()
print()
print('class balance (0=benign, 1=malignant):')
print(counts)
print('malignant fraction: {:.3f}'.format(df['target'].mean()))   # ~0.37 -> mild imbalance

# ---- Summary statistics for a handful of representative features ----
df[['mean radius', 'mean texture', 'mean area', 'mean concavity']].describe()

### 2.1 Visual EDA

Two questions drive the plots below: (1) which features separate the classes, and (2) how redundant are the features with each other? The answers shape our feature-engineering and model choices.

In [ ]:
# Correlation heatmap over the 10 'mean' features + target. A full 30x30 grid is unreadable,
# so we focus on the mean-* block, which is where most signal lives.
mean_cols = [c for c in feature_names if c.startswith('mean')]
corr = df[mean_cols + ['target']].corr()                   # Pearson correlation matrix

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, cbar_kws={'shrink': 0.8}, annot_kws={'size': 7})
plt.title('Correlation of mean-* features with each other and the target')
plt.tight_layout()
plt.show()

# Reading the last row/column (correlation WITH target): mean concave points, mean concavity,
# and mean radius/perimeter/area all correlate strongly positive with malignancy. Several of
# the size features (radius/perimeter/area) are ~0.99 correlated with EACH OTHER -> heavy
# multicollinearity, which is exactly why tree models or regularized linear models help.

In [ ]:
# Distribution of a few high-signal features split by class. If the two class distributions
# barely overlap, that feature is an easy, strong separator.
key_feats = ['mean concave points', 'mean radius', 'mean texture', 'mean smoothness']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, feat in zip(axes, key_feats):
    sns.boxplot(data=df, x='target', y=feat, ax=ax, hue='target',
                palette='coolwarm', legend=False)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['benign', 'malignant'])
    ax.set_xlabel('')
    ax.set_title(feat)
plt.suptitle('Feature distributions by class (0=benign, 1=malignant)')
plt.tight_layout()
plt.show()

# 'mean concave points' and 'mean radius' show a large gap between the boxes -> strongly
# predictive. 'mean smoothness' and 'mean texture' overlap much more -> weaker on their own.

## 3. Feature engineering (inside the pipeline)

All 30 inputs are numeric with no missing values, so the mandatory preprocessing is just **scaling** (needed by the linear and SVM models). On top of that we add three domain-motivated features:

- **area / perimeter** -- a compactness-like shape ratio.
- **worst radius / mean radius** -- how much the worst-case cell dwarfs the average (a proxy for irregularity).
- **concavity * area** -- an interaction term: tumors that are both large *and* concave are the worrying ones.

Crucially, every transform lives inside a scikit-learn `Pipeline`, so the scaler is fit on the training fold only during cross-validation -- this prevents data leakage from validation/test rows into preprocessing.

In [ ]:
# Separate the design matrix X from the target y. We keep X as a plain NumPy array so the
# engineered-feature step below can address columns by their fixed positional index.
X = df[feature_names].values                   # (569, 30)
y = df['target'].values                        # (569,)

# ---- Hold out a TEST set up front (25%) and never touch it until final evaluation. ----
# stratify=y preserves the ~37% malignant ratio in both splits so CV scores are representative.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)
print('train:', X_train.shape, ' test:', X_test.shape)


# ---- Feature engineering as a reusable, leakage-safe pipeline step ----
# Column indices are fixed by sklearn's feature ordering:
#   0 = mean radius, 2 = mean perimeter, 3 = mean area, 6 = mean concavity, 20 = worst radius.
def add_engineered_features(X):
    X = np.asarray(X, dtype=float)
    area_per_perimeter     = (X[:, 3] / X[:, 2]).reshape(-1, 1)   # shape ratio
    worst_over_mean_radius = (X[:, 20] / X[:, 0]).reshape(-1, 1)  # worst-vs-average irregularity
    concavity_x_area       = (X[:, 6] * X[:, 3]).reshape(-1, 1)   # size-and-concavity interaction
    return np.hstack([X, area_per_perimeter, worst_over_mean_radius, concavity_x_area])

engineered_names = ['area_per_perimeter', 'worst_over_mean_radius', 'concavity_x_area']
all_feature_names = feature_names + engineered_names   # names AFTER the FE step, in order

# Preprocessing bundle used by EVERY model below: add features, THEN standardize.
# Wrapping this in a Pipeline means scaling stats are learned on each CV train fold only,
# so there is no information leak from validation/test data.
def make_pipeline(estimator):
    return Pipeline([
        ('fe',    FunctionTransformer(add_engineered_features)),  # 30 -> 33 features
        ('scale', StandardScaler()),                              # zero mean / unit variance
        ('clf',   estimator),                                     # the model plugged in last
    ])

# Show the engineered columns for one example row to prove the step works.
print('engineered example (first train row):',
      np.round(add_engineered_features(X_train[:1])[0, -3:], 3))

## 4. Model selection via cross-validation

We compare four model families -- logistic regression, random forest, gradient boosting, and an RBF support vector machine -- under identical preprocessing, using 5-fold stratified cross-validation on the training set. Scoring on ROC AUC gives us a fair, threshold-free leaderboard before we commit to tuning one model. (The SVM needs no `probability=True` here: the AUC scorer ranks samples using its `decision_function`.)

In [ ]:
# Four candidate models spanning linear, kernel, and tree-ensemble families.
candidates = {
    'LogisticRegression': LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
    'RandomForest':       RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    'GradientBoosting':   GradientBoostingClassifier(random_state=RANDOM_STATE),
    'SVC(RBF)':           SVC(kernel='rbf', random_state=RANDOM_STATE),
}

# 5-fold stratified CV on the TRAIN set only. We score with ROC AUC because it is
# threshold-independent and robust to the class imbalance -> our primary selection metric.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, est in candidates.items():
    scores = cross_val_score(make_pipeline(est), X_train, y_train, cv=cv, scoring='roc_auc')
    rows.append({'model': name, 'cv_auc_mean': scores.mean(), 'cv_auc_std': scores.std()})

# Tidy results table, sorted best-first.
results = pd.DataFrame(rows).sort_values('cv_auc_mean', ascending=False).reset_index(drop=True)
print(results.to_string(index=False))

## 5. Hyperparameter tuning

All four models score very closely on AUC. We tune **GradientBoosting** with `GridSearchCV` over a small, sensible grid: it captures the feature interactions we engineered and handles the multicollinear size features gracefully. The search reuses the same stratified 5-fold CV and AUC scoring, then refits the best configuration on the full training set.

In [ ]:
# Small grid keeps the search fast (4 combinations x 5 folds). We vary the two knobs that
# most affect a gradient-boosted ensemble here: step size and tree depth (trees fixed at 200).
param_grid = {
    'clf__n_estimators':  [200],
    'clf__learning_rate': [0.05, 0.1],
    'clf__max_depth':     [2, 3],
}

grid = GridSearchCV(
    make_pipeline(GradientBoostingClassifier(random_state=RANDOM_STATE)),
    param_grid=param_grid, cv=cv, scoring='roc_auc', n_jobs=1)
grid.fit(X_train, y_train)                       # refits the best pipeline on ALL of X_train

print('best CV AUC : {:.4f}'.format(grid.best_score_))
print('best params :', grid.best_params_)

best_model = grid.best_estimator_                # the tuned, fully-fitted pipeline

## 6. Final evaluation on the held-out test set

Now -- and only now -- we touch the test set that was set aside at the start. We report a full classification report (precision/recall/F1 per class), the confusion matrix, the ROC curve with AUC, and the model's feature importances.

In [ ]:
# ---- FINAL, one-shot evaluation on the untouched held-out test set ----
y_pred  = best_model.predict(X_test)                          # hard 0/1 predictions
y_proba = best_model.predict_proba(X_test)[:, 1]              # P(malignant) for ROC/AUC

print('Classification report (1 = malignant, the positive class):')
print(classification_report(y_test, y_pred, target_names=['benign', 'malignant']))

test_auc = roc_auc_score(y_test, y_proba)
print('Test ROC AUC: {:.4f}'.format(test_auc))

In [ ]:
# Confusion matrix: the cell (true malignant, predicted benign) is the costly clinical error
# -- a missed cancer -- so we watch it closely.
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['benign', 'malignant'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion matrix (test set)')
plt.tight_layout()
plt.show()

# Report the two clinically meaningful counts explicitly.
tn, fp, fn, tp = cm.ravel()
print('false negatives (missed cancers):', int(fn))
print('false positives (false alarms) :', int(fp))

In [ ]:
# ROC curve traces true-positive vs false-positive rate across every threshold. Area under
# it (AUC) summarizes ranking quality; the dashed diagonal is a random classifier (AUC 0.5).
fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='#cc4444', lw=2, label='GradientBoosting (AUC = {:.3f})'.format(test_auc))
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='random')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curve (test set)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Which features drive the tuned model? GradientBoosting exposes feature_importances_
# aligned to the 33 columns AFTER our engineering step (30 originals + 3 engineered).
importances = best_model.named_steps['clf'].feature_importances_
imp = (pd.Series(importances, index=all_feature_names)
         .sort_values(ascending=False)
         .head(15))

plt.figure(figsize=(8, 6))
sns.barplot(x=imp.values, y=imp.index, hue=imp.index, palette='viridis', legend=False)
plt.xlabel('Importance (impurity reduction)')
plt.title('Top 15 feature importances')
plt.tight_layout()
plt.show()

# Print engineered-feature importances to see whether our hand-built features earned their keep.
print('Engineered feature importances:')
for nm in engineered_names:
    print('  {:<24} {:.4f}'.format(nm, importances[all_feature_names.index(nm)]))

In [ ]:
# Persist the ENTIRE fitted pipeline (feature engineering + scaler + tuned model) as one
# artifact. Saving the whole pipeline -- not just the classifier -- means inference code can
# feed in raw 30-column data and get identical preprocessing, with zero risk of train/serve skew.
import os
out_path = os.path.join(os.getcwd(), 'breast_cancer_pipeline.joblib')
joblib.dump(best_model, out_path)
print('saved pipeline to:', out_path)

# Sanity check: reload it and confirm it reproduces the same test AUC.
reloaded = joblib.load(out_path)
reloaded_auc = roc_auc_score(y_test, reloaded.predict_proba(X_test)[:, 1])
print('reloaded model test AUC: {:.4f}'.format(reloaded_auc))

## 7. REPORT

_Summary of the project, to be read on its own._

**Objective.** Classify breast masses as malignant vs benign from 30 image-derived measurements, optimizing ROC AUC (with malignant recall as the clinically critical secondary metric).

**Data.** 569 samples, 30 numeric features, no missing values, ~37% malignant. Mild class imbalance handled via stratified splitting and AUC-based selection.

**Key findings from EDA.** Size features (radius / perimeter / area) and concavity-related features (mean concave points, mean concavity) separate the classes most clearly. The size features are almost perfectly correlated with one another, signalling strong multicollinearity -- a reason to prefer tree ensembles or regularized/scaled linear models.

**Model comparison.** Four families were cross-validated under identical preprocessing. All landed in a tight band of ~0.99 mean CV AUC, confirming the problem is well-separated; the RBF SVM and logistic regression edged ahead on CV AUC, with gradient boosting a hair behind. GradientBoosting was selected for tuning as a strong, interaction-aware model that we can also read feature importances from.

**Tuning + final result.** A small GridSearchCV refined the boosting hyperparameters (best: learning_rate 0.1, max_depth 3, 200 trees). On the untouched test set the tuned pipeline achieved **ROC AUC approximately 0.9998** with **malignant recall approximately 0.91** (5 false negatives, 0 false positives on 143 test cases -- see the printed classification report and confusion matrix for exact values).

**Key drivers.** The most important features are the worst-case and mean concavity / concave-points and size measurements; among the engineered terms `worst_over_mean_radius` earns a meaningful mid-tier importance (~0.05), while `area_per_perimeter` and `concavity_x_area` add smaller amounts -- confirming the hand-built irregularity ratio adds real signal on top of the raw inputs.

**Limitations.** (1) Single, relatively small (n=569) historical dataset from one source -- results may not transfer to other imaging pipelines or populations. (2) A handful of false negatives remain; in a real deployment we would lower the decision threshold to trade some precision for higher malignant recall, since a missed cancer is the costliest error. (3) Feature importances reflect association, not causation.

**Next steps.** Threshold tuning on the malignant class to hit a target sensitivity; calibration of predicted probabilities (e.g. isotonic/Platt); external validation on an independent cohort; and adding per-prediction explainability before any clinical use. The fitted pipeline is saved to `breast_cancer_pipeline.joblib` for reuse.